# Fine-tuning CamemBERT-NER on CV / Job Offer annotations

**Goal**: take ~200 annotated CVs + ~100 annotated job offers exported from Label Studio, fine-tune `Jean-Baptiste/camembert-ner` on the project-specific label set (PERSON_CANDIDATE, JOB_TITLE, COMPANY, SKILL, DEGREE, LANGUAGE, YEARS_EXP, LOCATION, EMAIL, PHONE), and ship the resulting checkpoint to the production NER pipeline.

**Hardware**: a CPU run will work but takes ~3h on 300 docs. A single GPU (T4 / RTX 3060) finishes in ~15 min.

**Inputs**:
- One or more Label Studio export JSON files (`labels/*.json`) produced by `backend/scripts/export_ner_annotations.py` + manual correction.

**Outputs**:
- A fine-tuned checkpoint at `OUTPUT_DIR` (defaults to `/srv/ai-realtime/models/camembert-ner-airealtime/`).
- A classification report (precision / recall / F1 per label) printed inline + saved to `OUTPUT_DIR/eval_report.json`.

**Deploy to production**: once trained, set `AI_REALTIME_CAMEMBERT_NER_MODEL=/srv/ai-realtime/models/camembert-ner-airealtime/` and `AI_REALTIME_NER_BACKEND=camembert`, then restart the API.

## 1. Install dependencies
Run once per environment. Comment out if you've already installed.

In [ ]:
# %pip install -q transformers>=4.40 datasets>=2.18 seqeval>=1.2 accelerate>=0.30 scikit-learn

## 2. Configuration
Tweak paths and hyperparameters here.

In [ ]:
from pathlib import Path

# Folder containing one or more Label Studio export JSONs.
LABELS_DIR = Path('../../labels')
OUTPUT_DIR = Path('/srv/ai-realtime/models/camembert-ner-airealtime')
BASE_MODEL = 'Jean-Baptiste/camembert-ner'

LABELS = [
    'PERSON_CANDIDATE', 'JOB_TITLE', 'COMPANY', 'SKILL',
    'DEGREE', 'LANGUAGE', 'YEARS_EXP', 'LOCATION', 'EMAIL', 'PHONE',
]
# BIO tag list: O + B-/I- per label.
TAGS = ['O'] + [f'{prefix}-{lbl}' for lbl in LABELS for prefix in ('B', 'I')]
TAG2ID = {t: i for i, t in enumerate(TAGS)}
ID2TAG = {i: t for t, i in TAG2ID.items()}

MAX_LENGTH = 256       # CamemBERT max is 512 — 256 covers most CV pages and saves memory
BATCH_SIZE = 8
LR = 3e-5
EPOCHS = 5
TEST_SIZE = 0.2
SEED = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'{len(TAGS)} BIO tags, base={BASE_MODEL}, out={OUTPUT_DIR}')

## 3. Load Label Studio exports
Each task has `data.text` + `annotations[0].result` with the corrected spans. We discard tasks that have no annotations (annotators skipped them).

In [ ]:
import json

def load_label_studio_export(path: Path) -> list[dict]:
    with path.open(encoding='utf-8') as f:
        tasks = json.load(f)
    docs = []
    for t in tasks:
        text = t.get('data', {}).get('text', '')
        anns = t.get('annotations') or []
        if not text or not anns:
            continue
        spans = []
        for res in anns[0].get('result', []):
            v = res.get('value', {})
            labs = v.get('labels') or []
            if not labs:
                continue
            spans.append({'start': v['start'], 'end': v['end'], 'label': labs[0]})
        docs.append({'text': text, 'spans': spans})
    return docs

documents = []
for p in sorted(LABELS_DIR.glob('*.json')):
    docs = load_label_studio_export(p)
    print(f'{p.name}: {len(docs)} docs')
    documents.extend(docs)
print(f'\nTotal: {len(documents)} documents, {sum(len(d["spans"]) for d in documents)} spans')

## 4. Tokenize + align labels to subword tokens
CamemBERT uses SentencePiece. A single word may be split into multiple subword tokens. The standard recipe:
- First subword of a span gets `B-LABEL`.
- Subsequent subwords get `I-LABEL`.
- Subwords outside any span get `O`.
- Special tokens (`<s>`, `</s>`, padding) get `-100` so the loss ignores them.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, add_prefix_space=True)

def align_labels(text: str, spans: list[dict]) -> dict:
    enc = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        return_offsets_mapping=True,
    )
    labels = []
    sorted_spans = sorted(spans, key=lambda s: s['start'])
    for tok_start, tok_end in enc['offset_mapping']:
        if tok_start == tok_end:  # special token
            labels.append(-100)
            continue
        tag = 'O'
        for s in sorted_spans:
            if tok_start >= s['start'] and tok_end <= s['end']:
                prefix = 'B' if tok_start == s['start'] else 'I'
                tag = f'{prefix}-{s["label"]}'
                break
        labels.append(TAG2ID.get(tag, TAG2ID['O']))
    enc.pop('offset_mapping')
    enc['labels'] = labels
    return enc

examples = [align_labels(d['text'], d['spans']) for d in documents]
print(f'{len(examples)} examples tokenized')
# Sanity check: label distribution on the first 1000 subwords
from collections import Counter
flat = [l for e in examples[:50] for l in e['labels'] if l != -100]
print('label distribution (sample):', Counter(ID2TAG[l] for l in flat).most_common(10))

## 5. Train / eval split

In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

train_examples, eval_examples = train_test_split(
    examples, test_size=TEST_SIZE, random_state=SEED
)
train_ds = Dataset.from_list(train_examples)
eval_ds = Dataset.from_list(eval_examples)
print(f'train={len(train_ds)} eval={len(eval_ds)}')

## 6. Training

In [ ]:
import numpy as np
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)
from seqeval.metrics import classification_report, f1_score

model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(TAGS),
    id2label=ID2TAG,
    label2id=TAG2ID,
    ignore_mismatched_sizes=True,  # base model has 4 labels, ours has 21
)

collator = DataCollatorForTokenClassification(tokenizer)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=2)
    true_tags, pred_tags = [], []
    for p_seq, l_seq in zip(preds, labels):
        cur_true, cur_pred = [], []
        for p, l in zip(p_seq, l_seq):
            if l == -100:
                continue
            cur_true.append(ID2TAG[int(l)])
            cur_pred.append(ID2TAG[int(p)])
        true_tags.append(cur_true)
        pred_tags.append(cur_pred)
    return {'f1': f1_score(true_tags, pred_tags)}

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    seed=SEED,
    logging_steps=20,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)
trainer.train()

## 7. Detailed per-label evaluation
Look at recall on `SKILL` and precision on `PERSON_CANDIDATE` — those are the two we have today's biggest issues with.

In [ ]:
preds_out = trainer.predict(eval_ds)
preds = np.argmax(preds_out.predictions, axis=2)
labels = preds_out.label_ids
true_tags, pred_tags = [], []
for p_seq, l_seq in zip(preds, labels):
    cur_true, cur_pred = [], []
    for p, l in zip(p_seq, l_seq):
        if l == -100:
            continue
        cur_true.append(ID2TAG[int(l)])
        cur_pred.append(ID2TAG[int(p)])
    true_tags.append(cur_true)
    pred_tags.append(cur_pred)

report = classification_report(true_tags, pred_tags, digits=3)
print(report)

import json
(OUTPUT_DIR / 'eval_report.txt').write_text(report)
(OUTPUT_DIR / 'eval_meta.json').write_text(json.dumps({
    'base_model': BASE_MODEL,
    'n_train': len(train_ds),
    'n_eval': len(eval_ds),
    'labels': LABELS,
    'macro_f1': float(f1_score(true_tags, pred_tags, average='macro')),
}, indent=2))

## 8. Save the fine-tuned model

In [ ]:
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f'Model saved to {OUTPUT_DIR}')

## 9. Deploy to production (OVH)

1. Copy the model directory to the production server:
   ```bash
   rsync -avz /srv/ai-realtime/models/camembert-ner-airealtime/ \
     ovh-prod:/srv/ai-realtime/models/camembert-ner-airealtime/
   ```
2. Point the API at the fine-tuned weights:
   ```bash
   export AI_REALTIME_CAMEMBERT_NER_MODEL=/srv/ai-realtime/models/camembert-ner-airealtime
   export AI_REALTIME_NER_BACKEND=camembert
   ```
3. Restart the API; the next CV ingest will use the new model.
4. Watch `extraction_method`, `parsed_profile.skill_terms`, and `parsed_profile.person_name` on the next 20 CVs to confirm the new labels stick.

**Roll back** by unsetting `AI_REALTIME_NER_BACKEND` (defaults to `spacy`).

## 10. Active learning loop (next iteration)

Once the v1 model is live, mine confusion to drive the next annotation batch:

- For each new CV, run inference and capture the *minimum softmax probability* across tokens predicted with a non-O label. Low-confidence tokens are the high-value annotation targets.
- Add the top-K low-confidence docs to the next Label Studio batch (script in `backend/scripts/export_ner_annotations.py` already supports `--count`).
- Retrain with the merged dataset. Expect ~+2 F1 per 100 freshly annotated docs, plateauing around 0.92–0.94 macro-F1 for this label set.